# Latihan Mandiri - Classical Problems

**Nama:** Afelia Nasyifa  
**NIM:** 2411513010  
**Mata Kuliah:** Pemrograman Paralel  

## 1. Dining Philosophers dengan jumlah filsuf berbeda

Saya pakai cara arbitrator seperti di modul, yaitu `Semaphore(N-1)`. Jadi, tidak semua filsuf bisa berebut garpu pada saat yang sama. Setiap filsuf makan 20 kali, dengan waktu makan dan berpikir yang sama untuk setiap pengujian. Throughput dihitung dari total kegiatan makan dibagi waktu eksekusi.

Hasil waktunya bisa sedikit berubah setiap kali kode dijalankan karena penjadwalan thread.

In [1]:
import threading
import time

PUTARAN = 20
WAKTU_MAKAN = 0.012
WAKTU_BERPIKIR = 0.004

def uji_dining(n):
    garpu = [threading.Lock() for _ in range(n)]
    arbitrator = threading.Semaphore(n - 1)
    jumlah_makan = [0] * n

    def filsuf(i):
        kiri = garpu[i]
        kanan = garpu[(i + 1) % n]
        for _ in range(PUTARAN):
            with arbitrator:
                with kiri:
                    with kanan:
                        time.sleep(WAKTU_MAKAN)
                        jumlah_makan[i] += 1
            time.sleep(WAKTU_BERPIKIR)

    daftar_thread = [threading.Thread(target=filsuf, args=(i,)) for i in range(n)]
    mulai = time.perf_counter()
    for t in daftar_thread:
        t.start()
    for t in daftar_thread:
        t.join()
    durasi = time.perf_counter() - mulai
    total = sum(jumlah_makan)
    assert total == n * PUTARAN
    return {'N': n, 'Total makan': total, 'Waktu (detik)': durasi,
            'Throughput (makan/detik)': total / durasi}

hasil_dining = [uji_dining(n) for n in (3, 7, 10)]
print(f"{'N':>3} | {'Total makan':>11} | {'Waktu (s)':>10} | {'Throughput (/s)':>16}")
print('-' * 53)
for h in hasil_dining:
    print(f"{h['N']:>3} | {h['Total makan']:>11} | {h['Waktu (detik)']:>10.3f} | {h['Throughput (makan/detik)']:>16.2f}")
print('Semua filsuf selesai makan, tidak ada deadlock.')

  N | Total makan |  Waktu (s) |  Throughput (/s)
-----------------------------------------------------
  3 |          60 |      0.733 |            81.84
  7 |         140 |      0.579 |           241.90
 10 |         200 |      0.494 |           404.50
Semua filsuf selesai makan, tidak ada deadlock.


In [2]:
# Bandingkan kenaikan throughput dengan kenaikan jumlah filsuf
awal = hasil_dining[0]['Throughput (makan/detik)']
for h in hasil_dining[1:]:
    rasio_n = h['N'] / hasil_dining[0]['N']
    rasio_t = h['Throughput (makan/detik)'] / awal
    print(f"N=3 ke N={h['N']}: jumlah filsuf {rasio_n:.2f}x, throughput {rasio_t:.2f}x")

N=3 ke N=7: jumlah filsuf 2.33x, throughput 2.96x
N=3 ke N=10: jumlah filsuf 3.33x, throughput 4.94x


**Pembahasan:** Throughput biasanya bertambah ketika filsuf ditambah, tetapi **tidak harus naik linear**. Jumlah garpu memang bertambah, namun tiap filsuf tetap membutuhkan dua garpu yang bisa sedang dipakai tetangganya. Selain itu, thread harus bergantian memakai lock dan semaphore. Saat jumlah thread bertambah, waktu menunggu dan penjadwalan juga ikut berpengaruh. Jadi, naiknya jumlah filsuf tidak otomatis membuat jumlah makan per detik meningkat dengan perbandingan yang sama. Angka tepatnya mengacu pada output percobaan di atas.

## 2. Readers-Writers dengan prioritas writer

Pada solusi *fair* di modul, reader dan writer sama-sama mengantre lewat satu pintu. Di sini pintunya saya ubah memakai `Condition` supaya reader baru langsung ditahan kalau ada writer yang sedang menunggu. Reader yang sudah masuk tetap boleh menyelesaikan bacaannya.

Variabel `writer_menunggu` menjadi penentunya: jika nilainya lebih dari nol, reader baru belum boleh mulai membaca.

In [3]:
import threading
import time

class ReaderWriterPrioritas:
    def __init__(self):
        self.kondisi = threading.Condition()
        self.reader_aktif = 0
        self.writer_aktif = False
        self.writer_menunggu = 0

    def mulai_baca(self):
        with self.kondisi:
            while self.writer_aktif or self.writer_menunggu > 0:
                self.kondisi.wait()
            self.reader_aktif += 1

    def selesai_baca(self):
        with self.kondisi:
            self.reader_aktif -= 1
            if self.reader_aktif == 0:
                self.kondisi.notify_all()

    def mulai_tulis(self):
        with self.kondisi:
            self.writer_menunggu += 1
            try:
                while self.reader_aktif > 0 or self.writer_aktif:
                    self.kondisi.wait()
                self.writer_aktif = True
            finally:
                self.writer_menunggu -= 1

    def selesai_tulis(self):
        with self.kondisi:
            self.writer_aktif = False
            self.kondisi.notify_all()

rw = ReaderWriterPrioritas()
data = {'nilai': 0}
log = []
log_mutex = threading.Lock()

def catat(pesan):
    with log_mutex:
        log.append(pesan)

def baca(nama, lama=0.06):
    rw.mulai_baca()
    try:
        catat(f'{nama} MULAI baca: {data["nilai"]}')
        time.sleep(lama)
        catat(f'{nama} SELESAI baca')
    finally:
        rw.selesai_baca()

def tulis(nama, lama=0.04):
    rw.mulai_tulis()
    try:
        data['nilai'] += 1
        catat(f'{nama} MULAI tulis: {data["nilai"]}')
        time.sleep(lama)
        catat(f'{nama} SELESAI tulis')
    finally:
        rw.selesai_tulis()

In [4]:
# Skenario: reader pertama sudah membaca, lalu writer mengantre,
# setelah itu reader kedua mencoba masuk.
t1 = threading.Thread(target=baca, args=('Reader-1', 0.18))
t2 = threading.Thread(target=tulis, args=('Writer-1', 0.06))
t3 = threading.Thread(target=baca, args=('Reader-2', 0.02))

t1.start()
time.sleep(0.03)
t2.start()

# Pastikan writer betul-betul tercatat menunggu sebelum Reader-2 datang.
batas = time.perf_counter() + 2
while time.perf_counter() < batas:
    with rw.kondisi:
        if rw.writer_menunggu > 0:
            break
    time.sleep(0.001)
else:
    raise RuntimeError('Writer belum masuk antrean.')

t3.start()
for t in (t1, t2, t3):
    t.join(timeout=3)
    assert not t.is_alive(), 'Ada thread yang belum selesai.'

for baris in log:
    print(baris)
assert log.index('Writer-1 MULAI tulis: 1') < log.index('Reader-2 MULAI baca: 1')
print('Hasil: Reader-2 baru masuk setelah Writer-1 selesai.')

Reader-1 MULAI baca: 0
Reader-1 SELESAI baca
Writer-1 MULAI tulis: 1
Writer-1 SELESAI tulis
Reader-2 MULAI baca: 1
Reader-2 SELESAI baca
Hasil: Reader-2 baru masuk setelah Writer-1 selesai.


### Uji risiko starvation untuk reader

Sekarang saya buat beberapa writer datang berurutan, lalu satu reader menyusul. Saya cek apakah reader harus menunggu writer yang sudah mengantre. Pengujian ini dibatasi, jadi bukan simulasi writer tanpa henti.

In [5]:
rw2 = ReaderWriterPrioritas()
log2 = []
lock_log2 = threading.Lock()

# Reader awal sengaja menahan akses sebentar agar writer bisa berkumpul di antrean.
def reader_awal():
    rw2.mulai_baca()
    time.sleep(0.16)
    rw2.selesai_baca()

def writer_beruntun(i):
    rw2.mulai_tulis()
    try:
        with lock_log2:
            log2.append(f'Writer-{i}')
        time.sleep(0.025)
    finally:
        rw2.selesai_tulis()

def reader_baru():
    mulai_tunggu = time.perf_counter()
    rw2.mulai_baca()
    try:
        tunggu = time.perf_counter() - mulai_tunggu
        with lock_log2:
            log2.append('Reader-baru')
        print(f'Reader baru menunggu {tunggu:.3f} detik sebelum bisa membaca.')
    finally:
        rw2.selesai_baca()

awal = threading.Thread(target=reader_awal)
awal.start()
time.sleep(0.02)
writer_threads = [threading.Thread(target=writer_beruntun, args=(i,)) for i in range(1, 7)]
for t in writer_threads:
    t.start()

batas = time.perf_counter() + 2
while time.perf_counter() < batas:
    with rw2.kondisi:
        if rw2.writer_menunggu == len(writer_threads):
            break
    time.sleep(0.001)
else:
    raise RuntimeError('Tidak semua writer masuk antrean.')

reader_thread = threading.Thread(target=reader_baru)
reader_thread.start()
for t in [awal, *writer_threads, reader_thread]:
    t.join(timeout=3)
    assert not t.is_alive(), 'Thread belum selesai.'

print('Urutan layanan:', ' -> '.join(log2))
assert log2[-1] == 'Reader-baru'
print('Reader mendapat giliran setelah semua writer yang sudah mengantre selesai.')

Reader baru menunggu 0.290 detik sebelum bisa membaca.
Urutan layanan: Writer-1 -> Writer-6 -> Writer-4 -> Writer-5 -> Writer-2 -> Writer-3 -> Reader-baru
Reader mendapat giliran setelah semua writer yang sudah mengantre selesai.


**Pembahasan:** Hasil uji pertama menunjukkan reader yang datang belakangan tidak bisa mendahului writer yang sudah menunggu. Pada uji kedua, reader baru harus menunggu enam writer selesai lebih dulu. Kalau writer terus berdatangan tanpa jeda, reader bisa terus tertahan dan berisiko starvation. Dalam percobaan ini reader akhirnya tetap bisa masuk karena jumlah writer dibatasi, jadi hasilnya menunjukkan risiko starvation, bukan membuktikan reader benar-benar menunggu selamanya.

## 3. Sleeping Barber dengan 2 tukang cukur

Saya gunakan satu `queue.Queue(maxsize=3)` sebagai ruang tunggu bersama, lalu buat dua thread tukang cukur. Keduanya mengambil pelanggan dari antrean yang sama. Jadi, tukang cukur yang lebih dulu senggang akan mengambil pelanggan berikutnya.

Masing-masing pelanggan punya `Event` sendiri supaya pemberitahuan selesai tidak tertukar dengan pelanggan lain.

In [6]:
import threading
import queue
import time

JUMLAH_TUKANG_CUKUR = 2
KURSI_TUNGGU = 3
JUMLAH_PELANGGAN = 12
LAMA_CUKUR = 0.12

antrean = queue.Queue(maxsize=KURSI_TUNGGU)
log_barber = []
mutex_log = threading.Lock()
selesai_per_pelanggan = {}


def cetak_log(pesan):
    with mutex_log:
        log_barber.append(pesan)
        print(pesan)

def tukang_cukur(nomor):
    while True:
        pelanggan = antrean.get()  # kosong -> tidur / menunggu
        if pelanggan is None:
            antrean.task_done()
            break
        id_pelanggan, sinyal_selesai = pelanggan
        cetak_log(f'Tukang cukur-{nomor} mulai melayani pelanggan-{id_pelanggan}')
        time.sleep(LAMA_CUKUR)
        cetak_log(f'Tukang cukur-{nomor} selesai melayani pelanggan-{id_pelanggan}')
        sinyal_selesai.set()
        antrean.task_done()

pekerja = [threading.Thread(target=tukang_cukur, args=(i,))
           for i in range(1, JUMLAH_TUKANG_CUKUR + 1)]
for t in pekerja:
    t.start()

jumlah_diterima = 0
jumlah_ditolak = 0
for i in range(1, JUMLAH_PELANGGAN + 1):
    event = threading.Event()
    try:
        antrean.put_nowait((i, event))
        selesai_per_pelanggan[i] = event
        jumlah_diterima += 1
        cetak_log(f'Pelanggan-{i} masuk antrean')
    except queue.Full:
        jumlah_ditolak += 1
        cetak_log(f'Pelanggan-{i} pergi karena ruang tunggu penuh')
    time.sleep(0.015)

antrean.join()  # tunggu seluruh pelanggan diterima selesai dipotong
for _ in pekerja:
    antrean.put(None)  # hentikan kedua tukang cukur setelah antrean kosong
antrean.join()
for t in pekerja:
    t.join(timeout=3)
    assert not t.is_alive()

assert all(e.is_set() for e in selesai_per_pelanggan.values())
assert jumlah_diterima + jumlah_ditolak == JUMLAH_PELANGGAN
print(f'\nDiterima: {jumlah_diterima}, ditolak: {jumlah_ditolak}, total: {JUMLAH_PELANGGAN}')
print('Kedua tukang cukur selesai bertugas, semua pelanggan diterima sudah dilayani.')

Pelanggan-1 masuk antrean
Tukang cukur-1 mulai melayani pelanggan-1
Pelanggan-2 masuk antrean
Tukang cukur-2 mulai melayani pelanggan-2
Pelanggan-3 masuk antrean
Pelanggan-4 masuk antrean
Pelanggan-5 masuk antrean
Pelanggan-6 pergi karena ruang tunggu penuh
Pelanggan-7 pergi karena ruang tunggu penuh
Pelanggan-8 pergi karena ruang tunggu penuh
Tukang cukur-1 selesai melayani pelanggan-1
Tukang cukur-1 mulai melayani pelanggan-3
Pelanggan-9 masuk antrean
Tukang cukur-2 selesai melayani pelanggan-2
Tukang cukur-2 mulai melayani pelanggan-4
Pelanggan-10 masuk antrean
Pelanggan-11 pergi karena ruang tunggu penuh
Pelanggan-12 pergi karena ruang tunggu penuh


Tukang cukur-1 selesai melayani pelanggan-3
Tukang cukur-1 mulai melayani pelanggan-5
Tukang cukur-2 selesai melayani pelanggan-4
Tukang cukur-2 mulai melayani pelanggan-9
Tukang cukur-1 selesai melayani pelanggan-5
Tukang cukur-1 mulai melayani pelanggan-10
Tukang cukur-2 selesai melayani pelanggan-9


Tukang cukur-1 selesai melayani pelanggan-10

Diterima: 7, ditolak: 5, total: 12
Kedua tukang cukur selesai bertugas, semua pelanggan diterima sudah dilayani.


**Pembahasan:** Dua tukang cukur bisa bekerja bersamaan karena keduanya mengambil pelanggan dari satu antrean yang sama. `Queue` menjaga supaya satu pelanggan hanya diambil oleh satu tukang cukur. Saat antrean kosong, tukang cukur menunggu. Kalau antrean penuh saat ada pelanggan baru, pelanggan itu langsung pergi. Pada simulasi ini kedatangan pelanggan dibuat lebih cepat daripada waktu cukur agar kondisi antrean penuh juga terlihat. Kapasitas 3 yang dipakai di kode adalah kursi tunggu, di luar dua pelanggan yang sedang dicukur.

## 4. Refleksi singkat

Menurut saya, Producer-Consumer paling dekat dengan masalah yang bisa ditemui saat membuat software, terutama sistem monitoring berbasis kamera. Contohnya, kamera terus mengambil frame kendaraan sebagai *producer*, sedangkan program deteksi kendaraan menjadi *consumer* yang memproses frame tersebut. Kalau frame masuk lebih cepat daripada kemampuan pemrosesan, antrean bisa menumpuk dan hasil monitoring jadi terlambat. Masalah ini bisa ditangani dengan `Queue` yang kapasitasnya dibatasi. Dengan begitu, proses pengambilan gambar dan deteksi tidak perlu saling menunggu setiap saat. Menurut saya, contoh ini cukup nyata karena saat mengembangkan sistem, kita bukan cuma memikirkan hasil deteksi, tetapi juga bagaimana data masuk dan diproses dengan lancar tanpa membuat sistem kewalahan.